In [ ]:
# importing modules
from datasets import load_dataset

In [ ]:
# loading the dataset
dataset = load_dataset("rajpurkar/squad")
dataset

DatasetDict({
    train: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 87599
    })
    validation: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 10570
    })
})

In [ ]:
# process function to add end of text (unique to gpt2)
def add_end_of_text(example):
  example["question"] = example["question"] + "<|endoftext|>"
  return example

dataset = dataset.remove_columns(["id", "title", "context", "answers"])
dataset = dataset.map(add_end_of_text)
dataset["train"]["question"][:5]

['To whom did the Virgin Mary allegedly appear in 1858 in Lourdes France?<|endoftext|>',
 'What is in front of the Notre Dame Main Building?<|endoftext|>',
 'The Basilica of the Sacred heart at Notre Dame is beside to which structure?<|endoftext|>',
 'What is the Grotto at Notre Dame?<|endoftext|>',
 'What sits on top of the Main Building at Notre Dame?<|endoftext|>']

In [ ]:
# importing tokenizer
from transformers import AutoTokenizer
model = "distilgpt2"
tokenizer = AutoTokenizer.from_pretrained(model, use_fast=True)

In [ ]:
# converting a sample sentence to tokens
sequence = ("This tokenizer is being applied in CS197 at"
            "Harvard.<|endoftext|>")
tokens = tokenizer.tokenize(sequence)
print(tokens)

['This', 'Ġtoken', 'izer', 'Ġis', 'Ġbeing', 'Ġapplied', 'Ġin', 'ĠCS', '197', 'Ġat', 'Har', 'vard', '.', '<|endoftext|>']


In [ ]:
# indexing or converting the tokens to numbers
ids = tokenizer.convert_tokens_to_ids(tokens)
ids

[1212,
 11241,
 7509,
 318,
 852,
 5625,
 287,
 9429,
 24991,
 379,
 13587,
 10187,
 13,
 50256]

In [ ]:
# automatic chaining the operations above when the tokenizer is directly called
sequence = ("This tokenizer is being applied in CS197 at"
            "Harvard.<|endoftext|>")
tokenizer(sequence)

{'input_ids': [1212, 11241, 7509, 318, 852, 5625, 287, 9429, 24991, 379, 13587, 10187, 13, 50256], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}

In [ ]:
# tokenizing the dataset
def tokenize_function(example):
  return tokenizer(example["question"], truncation=True)

tokenized_dataset = dataset.map(tokenize_function, batched=True, num_proc=4, remove_columns=["question"])
tokenized_dataset

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask'],
        num_rows: 87599
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask'],
        num_rows: 10570
    })
})

In [ ]:
# concatenating different chunks together to have equal length
block_size = 128

def group_texts(examples):
    # repeat concatenation for input_ids and other keys
    concatenated_examples = {k: sum(examples[k], []) for k in
                            examples.keys()}
    total_length = len(concatenated_examples[list(examples.keys())[0]])
    total_length = (total_length // block_size) * block_size

    # populate each of input_ids and other keys
    result = {
        k: [t[i : i + block_size] for i in range(0,
            total_length, block_size)]
        for k, t in concatenated_examples.items()
    }
    # add labels because we'll need it as the output
    result["labels"] = result["input_ids"].copy()
    return result

lm_datasets = tokenized_dataset.map(
    group_texts,
    batched=True,
    batch_size=1000,
    num_proc=4,
)

Map (num_proc=4):   0%|          | 0/87599 [00:00<?, ? examples/s]

Map (num_proc=4):   0%|          | 0/10570 [00:00<?, ? examples/s]

In [ ]:
# printing the processed chunks or tokens
print(lm_datasets['train']['input_ids'][0])

[2514, 4150, 750, 262, 5283, 5335, 7910, 1656, 287, 1248, 3365, 287, 406, 454, 8906, 4881, 30, 50256, 2061, 318, 287, 2166, 286, 262, 23382, 20377, 8774, 11819, 30, 50256, 464, 32520, 3970, 286, 262, 17380, 2612, 379, 23382, 20377, 318, 13970, 284, 543, 4645, 30, 50256, 2061, 318, 262, 10299, 33955, 379, 23382, 20377, 30, 50256, 2061, 10718, 319, 1353, 286, 262, 8774, 11819, 379, 23382, 20377, 30, 50256, 2215, 750, 262, 3059, 349, 3477, 11175, 286, 23382, 288, 480, 2221, 12407, 30, 50256, 2437, 1690, 318, 23382, 20377, 338, 262, 39296, 1754, 3199, 30, 50256, 2061, 318, 262, 4445, 3710, 3348, 379, 23382, 20377, 1444, 30, 50256, 2437, 867, 3710, 1705, 9473, 389, 1043, 379, 23382, 20377, 30, 50256, 818, 644, 614, 750, 262, 3710, 3348]


In [ ]:
# decoding from indeces to words
tokenizer.decode(lm_datasets['train']['input_ids'][0])

"To whom did the Virgin Mary allegedly appear in 1858 in Lourdes France?<|endoftext|>What is in front of the Notre Dame Main Building?<|endoftext|>The Basilica of the Sacred heart at Notre Dame is beside to which structure?<|endoftext|>What is the Grotto at Notre Dame?<|endoftext|>What sits on top of the Main Building at Notre Dame?<|endoftext|>When did the Scholastic Magazine of Notre dame begin publishing?<|endoftext|>How often is Notre Dame's the Juggler published?<|endoftext|>What is the daily student paper at Notre Dame called?<|endoftext|>How many student news papers are found at Notre Dame?<|endoftext|>In what year did the student paper"

In [ ]:
# selecting a few samples of dataset to train in short time
small_train_dataset = \
    lm_datasets["train"].shuffle(seed=42).select(range(100))
small_eval_dataset = \
    lm_datasets["validation"].shuffle(seed=42).select(range(100))

In [ ]:
# importing model and trainer class
from transformers import AutoModelForCausalLM, TrainingArguments, Trainer

model = AutoModelForCausalLM.from_pretrained("distilgpt2")

model.safetensors: reconstructing file:   0%|          |  0.00B /  353MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [ ]:
# defining training arguments and training
training_args = TrainingArguments(
    "distilgpt2-squad-simeon",
    eval_strategy="epoch",
    learning_rate=2e-5,
    weight_decay=0.01,
    push_to_hub=True,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=small_train_dataset,
    eval_dataset=small_eval_dataset,
)

trainer.train()

Epoch,Training Loss,Validation Loss
1,No log,3.379891
2,No log,3.355599
3,No log,3.347948


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=39, training_loss=3.372314453125, metrics={'train_runtime': 58.8416, 'train_samples_per_second': 5.098, 'train_steps_per_second': 0.663, 'total_flos': 9798628147200.0, 'train_loss': 3.372314453125, 'epoch': 3.0})

In [ ]:
# importing metrics: a lower perplexity is better.
import math
eval_results = trainer.evaluate()
print(f"Perplexity: {math.exp(eval_results['eval_loss']):.2f}")

Training Loss,Validation Loss,Epoch
No log,3.462316,3


Perplexity: 31.89


In [ ]:
# uploading final model and tokenizer to the hub, but already done with save_to_hub
tokenizer.save_pretrained("Simeon-Krah/distilgpt2-squad-simeon")
model.push_to_hub("Simeon-Krah/distilgpt2-squad-simeon")

README.md:   0%|          | 0.00/5.17k [00:00<?, ?B/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...fxrhemg/model.safetensors:   5%|4         | 15.9MB /  328MB            

Validating                    :           |   0%            

No files have been modified since last commit. Skipping to prevent empty commit.


CommitInfo(commit_url='https://huggingface.co/Simeon-Krah/distilgpt2-squad-simeon/commit/75b2b2990226b5702a2f575546a74b399e9ee9e0', commit_message='Upload model', commit_description='', oid='75b2b2990226b5702a2f575546a74b399e9ee9e0', pr_url=None, repo_url=RepoUrl('https://huggingface.co/Simeon-Krah/distilgpt2-squad-simeon', endpoint='https://huggingface.co', repo_type='model', repo_id='Simeon-Krah/distilgpt2-squad-simeon'), pr_revision=None, pr_num=None)

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

# Load the saved model and tokenizer from the local directory instead of the broken Hub repo
try:
    model = AutoModelForCausalLM.from_pretrained("./distilgpt2-squad-simeon")
    tokenizer = AutoTokenizer.from_pretrained("./distilgpt2-squad-simeon")
    print("Successfully loaded the locally trained model and tokenizer!")
except Exception as e:
    print(f"Local loading failed with: {e}. Falling back to default distilgpt2 tokenizer and local model:")
    # Fallback if local path is different
    tokenizer = AutoTokenizer.from_pretrained("distilgpt2")

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

Successfully loaded the locally trained model and tokenizer!


In [ ]:
# generating with the trained model
from transformers import AutoModelForCausalLM, AutoTokenizer

# Ensure we use the locally trained model and its matching tokenizer
# (Alternatively, use the original 'distilgpt2' tokenizer)
#tokenizer = AutoTokenizer.from_pretrained("distilgpt2")

start_text = ("A speedrun is a playthrough of a video game, "
              "or section of a video game, with the goal of "
              "completing it as fast as possible. Speedruns "
              "often follow planned routes, which may incorporate sequence "
              "breaking, and might exploit glitches that allow sections to "
              "be skipped or completed more quickly than intended. ")

prompt = "What is the"
inputs = tokenizer(
     start_text + prompt,
     add_special_tokens=False,
     return_tensors="pt").to(model.device)

prompt_length = len(tokenizer.decode(inputs["input_ids"][0]))
outputs = model.generate(
     inputs["input_ids"],
     max_length=150,  # increased slightly to allow for the long prompt
     do_sample=True,
     top_k=50,
     top_p=0.95,
     temperature=0.9,
     num_return_sequences=3,
     pad_token_id=tokenizer.eos_token_id)  # explicitly setting pad_token_id

generated = prompt + tokenizer.decode(outputs[0])[prompt_length:]

print(tokenizer.decode(outputs[0], skip_special_tokens=True))

A speedrun is a playthrough of a video game, or section of a video game, with the goal of completing it as fast as possible. Speedruns often follow planned routes, which may incorporate sequence breaking, and might exploit glitches that allow sections to be skipped or completed more quickly than intended. What is the best way to speedrun?
